# Gist Graph: train, evaluate and release on Colab

Trains the graph compressor (M4) and the flat-slot comparison (M2) on Qwen2.5-0.5B-Instruct,
evaluates both against the non-learned baselines, writes the results page and publishes the graph
model to the Hugging Face Hub.

**Runtime → Change runtime type → GPU.** A T4 works (fp16, about 2-3 h for everything); an L4 or
A100 is faster and uses bf16. Runs and caches live on Google Drive, so a disconnected session
resumes from its last checkpoint when you re-run the cells.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

## 1. Setup
Mount Drive for persistent runs, clone the repository and install it.

In [ ]:
from google.colab import drive

drive.mount("/content/drive")

WORK = "/content/drive/MyDrive/gistgraph"  # runs, caches and the exported model live here
REPO = "https://github.com/dhrrishitvdeka/gistgraph.git"
HF_REPO = "dhrrishitvdeka/gistgraph-qwen2.5-0.5b"

In [ ]:
import os

os.makedirs(f"{WORK}/runs", exist_ok=True)
os.makedirs(f"{WORK}/cache", exist_ok=True)
%cd /content
!test -d gistgraph || git clone --depth 1 {REPO}
%cd /content/gistgraph
!git pull --ff-only
!pip install -q -e ".[baselines]"
# keep runs and caches on Drive so they survive a disconnect
!rm -rf runs cache && ln -s {WORK}/runs runs && ln -s {WORK}/cache cache

## 2. Baselines
Truncation, extractive selection and LLMLingua-2, on the same 200 examples per dataset as the learned models.

In [ ]:
!python -m gistgraph baselines --config configs/experiments/m1_baselines.yaml data.n_eval=200

## 3. Train and evaluate the compressors
Graph first, then flat. Each run builds (or reuses) the teacher cache, trains 1000 steps with
checkpoints every 250, then evaluates at 2-5x compression. Re-running this cell resumes.
Watch `mean_degree` in the log: it should stay above zero (edges in use).

In [ ]:
EXP = "configs/experiments"
!python scripts/run_experiments.py {EXP}/m4_graph.yaml {EXP}/m2_flat.yaml

## 4. Results page

In [ ]:
!python -m gistgraph report runs/m1_baselines runs/m2_flat runs/m4_graph --out docs/results
!cp -r docs/results {WORK}/results
from IPython.display import Markdown, display

with open("docs/results/results.md") as f:
    display(Markdown(f.read()))

## 5. Export and try the model

In [ ]:
!python -m gistgraph export runs/m4_graph {WORK}/release/gistgraph-qwen2.5-0.5b
!cp docs/results/results.md {WORK}/release/gistgraph-qwen2.5-0.5b/RESULTS.md

In [ ]:
from gistgraph import GistGraph

gg = GistGraph.from_pretrained(f"{WORK}/release/gistgraph-qwen2.5-0.5b")
context = (
    "Marie Curie was born in Warsaw in 1867. She moved to Paris in 1891 to study physics. "
    "In 1903 she shared the Nobel Prize in Physics with Pierre Curie and Henri Becquerel."
)
memory = gg.compress(context, ratio=4)
print(f"{memory.n_tokens} tokens -> {memory.n_nodes} nodes ({memory.ratio:.1f}x)")
print(gg.answer("Where was Marie Curie born?", memory=memory))

## 6. Publish to the Hugging Face Hub
Log in with a **write** token from https://huggingface.co/settings/tokens, then upload. Check the
results above first: the model is published with whatever it achieved.

In [ ]:
from huggingface_hub import notebook_login

notebook_login()

In [ ]:
from huggingface_hub import HfApi

url = gg.push_to_hub(HF_REPO)
HfApi().upload_file(
    path_or_fileobj=f"{WORK}/release/gistgraph-qwen2.5-0.5b/RESULTS.md",
    path_in_repo="RESULTS.md",
    repo_id=HF_REPO,
)
print(url)

## 7. Bring the results back to the repository
Download `results.zip` and copy `docs/results/` into your checkout, then commit it.

In [ ]:
!cd docs && zip -qr /content/results.zip results
from google.colab import files

files.download("/content/results.zip")